## Carbon Emissions Predictor using Linear Regression

### Load the Training Dataset into Unity Catalog Volume

In [ ]:
%sql

CREATE VOLUME IF NOT EXISTS carbonops.bronze.ml_raw_files;

In [ ]:
import requests

# Define the current catalog
catalog_name = "carbonops"

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/bronze/ml_raw_files"

# List of files to download
files = ["carbon_emissions_dataset.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/Databricks-Job-Ready-Projects/refs/heads/main/Machine-Learning/Linear-Regression-Model/Data/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

### Split the dataset into training and testing sets

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
   
data = spark.read.format("csv").option("header", "true").load(f"/Volumes/{catalog_name}/bronze/ml_raw_files/carbon_emissions_dataset.csv")

# Clean and cast columns
data = data.dropna().select(
    col("year").cast("int"),
    col("month").cast("int"),
    col("electricity_kwh").cast("double"),
    col("diesel_litres").cast("double"),
    col("natural_gas_m3").cast("double"),
    col("lpg_kg").cast("double"),
    col("production_units").cast("double"),
    col("operating_hours").cast("double"),
    col("avg_temperature_c").cast("double"),
    col("total_emissions_tco2e").cast("double")
)

# Split into Training and Testing Datasets
splits = data.randomSplit([0.7, 0.3], seed=42)

train = splits[0]
test = splits[1]

print("Training Rows:", train.count(), "Testing Rows:", test.count())

### Creating an MLflow Experiment Function

In [ ]:
def train_carbon_emissions_model(training_data, test_data, maxIterations, regularization):

    import mlflow
    import mlflow.sklearn
    from mlflow.models import infer_signature
    from sklearn.linear_model import Ridge
    from sklearn.preprocessing import MinMaxScaler
    from sklearn.pipeline import Pipeline
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    import pandas as pd
    import numpy as np
    import time

    # Start an MLflow run
    with mlflow.start_run():

        numFeatures = [
            "year",
            "month",
            "electricity_kwh",
            "diesel_litres",
            "natural_gas_m3",
            "lpg_kg",
            "production_units",
            "operating_hours",
            "avg_temperature_c"
        ]

        # Convert Spark DataFrames to pandas
        train_pd = training_data.select(
            numFeatures + ["total_emissions_tco2e"]
        ).toPandas()

        test_pd = test_data.select(
            numFeatures + ["total_emissions_tco2e"]
        ).toPandas()

        X_train = train_pd[numFeatures]
        y_train = train_pd["total_emissions_tco2e"]

        X_test = test_pd[numFeatures]
        y_test = test_pd["total_emissions_tco2e"]

        # Create preprocessing and regression pipeline
        model = Pipeline([
            ("scaler", MinMaxScaler()),
            ("regressor", Ridge(
                alpha=regularization,
                max_iter=maxIterations,
                random_state=42
            ))
        ])

        # Log parameters
        mlflow.log_param("maxIter", maxIterations)
        mlflow.log_param("regParam", regularization)
        mlflow.log_param("model_type", "Ridge Linear Regression")

        # Train the model
        print("Training Carbon Emissions Linear Regression model...")

        model.fit(X_train, y_train)

        # Evaluate the model
        y_pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)

        print("MAE:", mae)
        print("RMSE:", rmse)
        print("R2 Score:", r2)

        # Log metrics
        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("RMSE", rmse)
        mlflow.log_metric("R2", r2)

        # Create model input example
        input_example = pd.DataFrame([{
            "year": 2026,
            "month": 6,
            "electricity_kwh": 50000.0,
            "diesel_litres": 2500.0,
            "natural_gas_m3": 1200.0,
            "lpg_kg": 500.0,
            "production_units": 10000.0,
            "operating_hours": 500.0,
            "avg_temperature_c": 32.0
        }])

        # Infer model signature from actual predictions
        output_example = model.predict(input_example)
        signature = infer_signature(input_example, output_example)

        # Log sklearn model with signature
        unique_model_name = "carbon_emissions_sklearn_" + str(int(time.time()))

        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path=unique_model_name,
            signature=signature,
            input_example=input_example
        )

        print("Experiment run complete with Carbon Emissions Regression model!")

### Train the Model and Log Metrics to MLflow

In [ ]:
train_carbon_emissions_model(
    training_data=train,
    test_data=test,
    maxIterations=1000,
    regularization=1.0
)

### Load the Registered Model from Unity Catalog and Make Predictions

In [ ]:
import mlflow
import pandas as pd

mlflow.set_registry_uri("databricks-uc")

persistedModel = mlflow.sklearn.load_model(
    "models:/<YOUR_CATALOG_NAME>.<YOUR_SCHEMA_NAME>.<YOUR_MODEL_NAME>/<YOUR_MODEL_VERSION>"
)

newData = pd.DataFrame([{
    "year": 2026,
    "month": 6,
    "electricity_kwh": 50000.0,
    "diesel_litres": 2500.0,
    "natural_gas_m3": 1200.0,
    "lpg_kg": 500.0,
    "production_units": 10000.0,
    "operating_hours": 500.0,
    "avg_temperature_c": 32.0
}])

newData["PredictedEmissions_tCO2e"] = persistedModel.predict(newData)

display(newData)